In [1]:
# Install packages
!pip -q install geopandas pyogrio


In [2]:
# Import tools
import io
import zipfile
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
from IPython.display import display

def sitekey(data):
    state = data["state"].astype(int).astype(str).str.zfill(2)
    county = data["county"].astype(int).astype(str).str.zfill(3)
    number = data["number"].astype(int).astype(str).str.zfill(4)
    return state+"-"+county+"-"+number


In [3]:
# Load air districts
url = ("https://services6.arcgis.com/x7ftScCDR8g2kVFB/arcgis/rest/services/"
     "Air_District_WFL1/FeatureServer/0/query?"
     "where=1%3D1&outFields=Air_District_Name&outSR=4326&f=geojson")
districts = gpd.read_file(url).to_crs("EPSG:4326")
districts = districts.rename(columns={"Air_District_Name":"district"}).dissolve(by="district",as_index=False)
bay = districts[districts["district"].str.contains("Bay Area",case=False,na=False)].copy()
assert len(bay)==1
display(districts[["district"]].sort_values("district").reset_index(drop=True))


,district
0,AMADOR COUNTY APCD
1,ANTELOPE VALLEY AQMD
2,BAY AREA AQMD
3,BUTTE COUNTY AQMD
4,CALAVERAS COUNTY APCD
5,COLUSA COUNTY APCD
6,EASTERN KERN APCD
7,EL DORADO COUNTY AQMD
8,FEATHER RIVER AQMD
9,GLENN COUNTY APCD


In [4]:
# Download air data
years = range(2018,2025)
frames = []
for year in years:
    print("Loading",year)
    url = f"https://aqs.epa.gov/aqsweb/airdata/daily_88101_{year}.zip"
    data = pd.read_csv(url,compression="zip",low_memory=False)
    data = data.rename(columns={"State Code":"state","County Code":"county","Site Num":"number","Date Local":"date","Arithmetic Mean":"pm","Latitude":"lat","Longitude":"lon"})
    data = data[pd.to_numeric(data["state"],errors="coerce")==6].copy()
    data["date"] = pd.to_datetime(data["date"])
    data["pm"] = pd.to_numeric(data["pm"],errors="coerce")
    data["site"] = sitekey(data)
    data = data[data["date"].dt.month.between(6,10)]
    frames.append(data.groupby(["site","date"],as_index=False).agg(pm=("pm","mean"),lat=("lat","first"),lon=("lon","first")))
air = pd.concat(frames,ignore_index=True)
air["year"] = air["date"].dt.year
assert set(air["year"].unique())==set(years)
print("Daily rows:",len(air))
print("Unique sites:",air["site"].nunique())
display(air.head())


Loading 2018
Loading 2019
Loading 2020
Loading 2021
Loading 2022
Loading 2023
Loading 2024
Daily rows: 108477
Unique sites: 145


,site,date,pm,lat,lon,year
0,06-001-0007,2018-06-01,6.633333,37.687526,-121.784217,2018
1,06-001-0007,2018-06-02,7.112500,37.687526,-121.784217,2018
2,06-001-0007,2018-06-03,6.837500,37.687526,-121.784217,2018
3,06-001-0007,2018-06-04,7.928261,37.687526,-121.784217,2018
4,06-001-0007,2018-06-05,7.041666,37.687526,-121.784217,2018


In [5]:
# Qualify monitoring sites
coverage = air.groupby(["site","year"],as_index=False).agg(days=("date","nunique"))
stats = coverage.groupby("site",as_index=False).agg(years=("year","nunique"),days=("days","median"),total=("days","sum"))
coords = air.groupby("site",as_index=False).agg(lat=("lat","first"),lon=("lon","first"),mean=("pm","mean"),max=("pm","max"))
sites = stats.merge(coords,on="site")
sites = sites[(sites["years"]>=5)&(sites["days"]>=40)].copy()
points = gpd.GeoDataFrame(sites,geometry=gpd.points_from_xy(sites["lon"],sites["lat"]),crs="EPSG:4326")
assert len(points)==104 and points["site"].is_unique
print("Qualified sites:",len(points))
display(sites.sort_values("total",ascending=False).head(10))


Qualified sites: 104


,site,years,days,total,lat,lon,mean,max
133,06-099-0006,7,153.0,1071,37.488317,-120.836008,12.295396,118.520834
81,06-065-8001,7,153.0,1071,33.999580,-117.416010,13.337940,73.894792
82,06-065-8005,7,153.0,1071,33.996360,-117.492400,14.393324,85.133333
139,06-111-0007,7,153.0,1071,34.210169,-118.870509,9.258536,36.337500
68,06-059-0007,7,153.0,1071,33.830620,-117.938450,11.188810,63.290278
140,06-111-0009,7,153.0,1070,34.404281,-118.809980,9.856057,40.020833
64,06-053-1003,7,153.0,1070,36.694261,-121.623271,6.783399,87.041666
26,06-027-1003,7,153.0,1069,36.487823,-117.871036,9.826542,174.629166
90,06-071-0306,7,153.0,1069,34.510961,-117.325540,10.114679,81.418750
142,06-111-2002,7,153.0,1068,34.276316,-118.683685,9.324382,34.929167


In [6]:
# Download smoke data
folder = Path("hms")
folder.mkdir(exist_ok=True)
base = "https://satepsanone.nesdis.noaa.gov/pub/FIRE/web/HMS/Smoke_Polygons/Shapefile/Annual_Bundles"
for year in years:
    print("Loading",year)
    response = requests.get(f"{base}/hms_smoke{year}.zip",timeout=180)
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as bundle:
        bundle.extractall(folder/str(year))


Loading 2018
Loading 2019
Loading 2020
Loading 2021
Loading 2022
Loading 2023
Loading 2024


In [7]:
# Match smoke polygons
levels = {"5":1,"5.0":1,"light":1,"16":2,"16.0":2,"medium":2,"21":3,"21.0":3,"heavy":3}
records = []
for year in years:
    print("Matching",year)
    shapefiles = list((folder/str(year)).rglob("*.shp"))
    assert len(shapefiles)>0
    frames = [gpd.read_file(file) for file in shapefiles]
    clouds = gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry="geometry",crs=frames[0].crs).to_crs("EPSG:4326")
    start = clouds["Start"].astype(str)
    date8 = start.str.extract(r"(\d{8})")[0]
    date7 = start.str.extract(r"(\d{7})")[0]
    clouds["date"] = pd.to_datetime(date8,format="%Y%m%d",errors="coerce")
    clouds["date"] = clouds["date"].fillna(pd.to_datetime(date7,format="%Y%j",errors="coerce"))
    clouds["level"] = clouds["Density"].astype(str).str.lower().str.strip().map(levels)
    clouds = clouds[clouds["date"].dt.month.between(6,10)].copy()
    clouds = clouds.cx[-124.6:-114.0,32.5:42.1]
    hit = gpd.sjoin(points[["site","geometry"]],clouds[["date","level","geometry"]],how="inner",predicate="intersects")
    records.append(hit.groupby(["site","date"],as_index=False).agg(level=("level","max")))
hits = pd.concat(records,ignore_index=True)
assert set(hits["level"].dropna().unique()).issubset({1,2,3})
print("Smoke station-days:",len(hits))
display(hits.head())


Matching 2018
Matching 2019
Matching 2020
Matching 2021
Matching 2022
Matching 2023
Matching 2024
Smoke station-days: 30607


,site,date,level
0,06-001-0007,2018-06-25,1
1,06-001-0007,2018-06-30,1
2,06-001-0007,2018-07-02,3
3,06-001-0007,2018-07-03,1
4,06-001-0007,2018-07-04,1


In [8]:
# Summarize target regions
terms = ["Bay Area","Sacramento","San Joaquin","South Coast"]
station = gpd.sjoin(points[["site","geometry"]],districts[["district","geometry"]],how="left",predicate="intersects")
station = station[["site","district"]].drop_duplicates("site")
days = air[air["site"].isin(points["site"])].merge(hits,on=["site","date"],how="left")
days["level"] = days["level"].fillna(0)
days = days.merge(station,on="site")
days["smoke"] = (days["level"]>0).astype(int)
days["medium"] = (days["level"]>=2).astype(int)
days["smokepm"] = days["pm"].where(days["level"]>0)
siteyear = days.groupby(["site","district","year"],as_index=False).agg(observed=("date","nunique"),smoke=("smoke","sum"),medium=("medium","sum"),mean=("smokepm","mean"),peak=("smokepm","max"))
siteyear["rate"] = siteyear["smoke"]/siteyear["observed"]
summary = siteyear.groupby("district",as_index=False).agg(sites=("site","nunique"),smoke=("smoke","median"),medium=("medium","median"),rate=("rate","median"),mean=("mean","mean"),peak=("peak","max"))
regions = summary[summary["district"].str.contains("|".join(terms),case=False,na=False)].sort_values("smoke",ascending=False)
assert len(regions)==4
display(regions.reset_index(drop=True).round(3))


,district,sites,smoke,medium,rate,mean,peak
0,SACRAMENTO METROPOLITAN AQMD,5,45.0,10.0,0.314,13.073,263.531
1,BAY AREA AQMD,16,41.0,8.0,0.270,11.597,167.704
2,SAN JOAQUIN VALLEY UNIFIED APCD,17,34.5,6.0,0.275,17.783,248.000
3,SOUTH COAST AQMD,18,18.0,3.0,0.180,13.887,130.938


In [9]:
# Load population data
folder = Path("tracts")
folder.mkdir(exist_ok=True)
tracturl = "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_06_tract.zip"
tractfile = folder/"tracts.zip"
response = requests.get(tracturl,timeout=180)
response.raise_for_status()
tractfile.write_bytes(response.content)
with zipfile.ZipFile(tractfile) as bundle:
    bundle.extractall(folder)
tracts = gpd.read_file(folder/"tl_2024_06_tract.shp").rename(columns={"GEOID":"tract","NAME":"name"})
popurl = ("https://www2.census.gov/programs-surveys/acs/summary_file/"
        "2024/table-based-SF/data/5YRData/acsdt5y2024-b01003.dat")
pop = pd.read_csv(popurl,sep="|",dtype={"GEO_ID":str})
pop = pop[pop["GEO_ID"].str.startswith("1400000US06")].copy()
pop["tract"] = pop["GEO_ID"].str[-11:]
pop = pop.rename(columns={"B01003_E001":"pop"})
pop["pop"] = pd.to_numeric(pop["pop"])
tracts = tracts.merge(pop[["tract","pop"]],on="tract")
print("California tracts:",len(tracts))
print("California population:",tracts["pop"].sum())
display(tracts[["tract","name","pop"]].head())


California tracts: 9129
California population: 39287377


,tract,name,pop
0,06001442700,4427,3068
1,06001442800,4428,3041
2,06037204920,2049.20,2274
3,06037205110,2051.10,3501
4,06037205120,2051.20,3252


In [10]:
# Build population target
tracts = tracts.to_crs("EPSG:3310")
bayproj = bay.to_crs("EPSG:3310")
tracts["area"] = tracts.geometry.area
baytracts = gpd.overlay(tracts[["tract","name","pop","area","geometry"]],bayproj[["geometry"]],how="intersection")
baytracts["share"] = (baytracts.geometry.area/baytracts["area"]).clip(upper=1)
baytracts["target"] = baytracts["pop"]*baytracts["share"]
assert baytracts["target"].sum()>0
print("Bay Area tracts:",len(baytracts))
print("Target population:",round(baytracts["target"].sum()))
display(baytracts[["tract","name","pop","share","target"]].head())


Bay Area tracts: 1758
Target population: 7409446


,tract,name,pop,share,target
0,06001442700,4427,3068,1.0,3068.0
1,06001442800,4428,3041,1.0,3041.0
2,06001442900,4429,7783,1.0,7783.0
3,06095252204,2522.04,5874,1.0,5874.0
4,06095252203,2522.03,4299,1.0,4299.0


In [11]:
# Assign target weights
projected = points.to_crs("EPSG:3310")
targets = baytracts[["tract","target","geometry"]].copy()
targets["geometry"] = targets.geometry.representative_point()
assigned = gpd.sjoin_nearest(targets,projected[["site","geometry"]],how="left",distance_col="distance")
weights = assigned.groupby("site",as_index=False).agg(pop=("target","sum"),tracts=("tract","nunique"))
target = points.merge(weights,on="site",how="left")
target["pop"] = target["pop"].fillna(0)
target["tracts"] = target["tracts"].fillna(0).astype(int)
target["weight"] = target["pop"]/target["pop"].sum()
assert abs(target["weight"].sum()-1)<1e-10 and int((target["weight"]>0).sum())==21
print("Weight sum:",target["weight"].sum())
print("Positive sites:",(target["weight"]>0).sum())
display(target[["site","lat","lon","pop","tracts","weight"]].sort_values("weight",ascending=False).head(10))


Weight sum: 1.0
Positive sites: 21


,site,lat,lon,pop,tracts,weight
78,06-075-0005,37.765946,-122.399044,1.109811e+06,309,0.149783
88,06-085-0005,37.348497,-121.894898,1.028125e+06,213,0.138759
83,06-081-1001,37.482934,-122.203370,7.634657e+05,179,0.103040
10,06-013-0002,37.936013,-122.026154,6.631098e+05,140,0.089495
89,06-085-0006,37.338135,-121.849783,6.107153e+05,138,0.082424
1,06-001-0009,37.743065,-122.169935,5.093820e+05,112,0.068748
93,06-095-0004,38.102507,-122.237976,4.632461e+05,123,0.062521
5,06-001-0015,37.701222,-121.903019,4.522440e+05,86,0.061036
94,06-097-0004,38.403765,-122.818294,4.261924e+05,108,0.057520
3,06-001-0012,37.793624,-122.263376,2.563530e+05,63,0.034598


In [12]:
# Build final tables
meta = target.merge(station,on="site",how="left")
meta["bay"] = meta.geometry.intersects(bay.geometry.iloc[0]).astype(int)
meta = meta[["site","lat","lon","district","bay","years","days","mean","max","pop","tracts","weight","geometry"]].copy()
meta = meta.sort_values(["bay","weight","site"],ascending=[False,False,True])
final = air[air["site"].isin(meta["site"])].merge(hits,on=["site","date"],how="left")
final["level"] = final["level"].fillna(0).astype(int)
order = sorted(meta["site"].tolist())
pmwide = final.pivot_table(index="date",columns="site",values="pm",aggfunc="mean").reindex(columns=order)
smokewide = final.pivot_table(index="date",columns="site",values="level",aggfunc="max").reindex(columns=order).fillna(0).astype(int)
vector = meta.set_index("site").reindex(order)["weight"].reset_index()
assert pmwide.shape==smokewide.shape and pmwide.shape[1]==104
assert list(pmwide.columns)==order and list(smokewide.columns)==order
print("PM matrix:",pmwide.shape)
print("Smoke matrix:",smokewide.shape)
print("Missing fraction:",round(pmwide.isna().mean().mean(),4))
display(meta.drop(columns="geometry").head(15))


PM matrix: (1071, 104)
Smoke matrix: (1071, 104)
Missing fraction: 0.1624


,site,lat,lon,district,bay,years,days,mean,max,pop,tracts,weight
78,06-075-0005,37.765946,-122.399044,BAY AREA AQMD,1,7,151.0,8.266521,147.316666,1.109811e+06,309,0.149783
88,06-085-0005,37.348497,-121.894898,BAY AREA AQMD,1,7,153.0,10.202912,120.520834,1.028125e+06,213,0.138759
83,06-081-1001,37.482934,-122.203370,BAY AREA AQMD,1,6,152.0,7.940847,124.133333,7.634657e+05,179,0.103040
10,06-013-0002,37.936013,-122.026154,BAY AREA AQMD,1,7,153.0,8.451498,120.358333,6.631098e+05,140,0.089495
89,06-085-0006,37.338135,-121.849783,BAY AREA AQMD,1,7,153.0,9.693689,123.112500,6.107153e+05,138,0.082424
1,06-001-0009,37.743065,-122.169935,BAY AREA AQMD,1,7,153.0,8.754826,167.704166,5.093820e+05,112,0.068748
93,06-095-0004,38.102507,-122.237976,BAY AREA AQMD,1,7,153.0,8.533517,152.995834,4.632461e+05,123,0.062521
5,06-001-0015,37.701222,-121.903019,BAY AREA AQMD,1,7,153.0,9.368566,123.809091,4.522440e+05,86,0.061036
94,06-097-0004,38.403765,-122.818294,BAY AREA AQMD,1,7,150.0,5.851697,124.316666,4.261924e+05,108,0.057520
3,06-001-0012,37.793624,-122.263376,BAY AREA AQMD,1,7,152.0,9.835384,160.316666,2.563530e+05,63,0.034598


In [13]:
# Save region files
from google.colab import files
meta.drop(columns="geometry").to_csv("sites.csv",index=False)
vector.to_csv("target.csv",index=False)
pmwide.to_csv("pm.csv")
smokewide.to_csv("smoke.csv")
siteyear.to_csv("siteyear.csv",index=False)
regions.to_csv("regions.csv",index=False)
bay[["district","geometry"]].to_file("bay.geojson",driver="GeoJSON")
names = ["sites.csv","target.csv","pm.csv","smoke.csv","siteyear.csv","regions.csv","bay.geojson"]
with zipfile.ZipFile("regions.zip","w") as bundle:
    for name in names:
        bundle.write(name)
files.download("regions.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>